# Text model

In [ ]:
# load model
import torch
from transformers import AutoProcessor, AutoModel

# I modified one file in tranformers to make it work:
# https://github.com/huggingface/transformers/pull/38637/files

model_clip = AutoModel.from_pretrained(
    "openai/clip-vit-base-patch32", torch_dtype=torch.float32
)  # be careful with alternative attention implementations

processor = AutoProcessor.from_pretrained(
    "openai/clip-vit-base-patch32",
)


In [2]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

outputs = model_clip(**inputs)
# logits_per_image = outputs.logits_per_image
# probs = logits_per_image.softmax(dim=1)
# most_likely_idx = probs.argmax(dim=1).item()
# most_likely_label = labels[most_likely_idx]
# print(f"Most likely label: {most_likely_label} with probability: {probs[0][most_likely_idx].item():.3f}")

In [3]:
outputs.keys()

odict_keys(['logits_per_image', 'logits_per_text', 'text_embeds', 'image_embeds', 'text_model_output', 'vision_model_output'])

In [4]:
for child in model_clip.named_children():
    print(child, end="\n###############################\n")

('text_model', CLIPTextTransformer(
  (embeddings): CLIPTextEmbeddings(
    (token_embedding): Embedding(49408, 512)
    (position_embedding): Embedding(77, 512)
  )
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (layer_norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (final

In [5]:
class TextModel(torch.nn.Module):
    def __init__(self, text_model, text_projection):
        super().__init__()
        self.text_model =text_model 
        self.text_projection = text_projection

    def forward(self, input_ids, attention_mask = None):
        output = self.text_model(input_ids, attention_mask)
        output = output.pooler_output
        output = self.text_projection(output)
        output = torch.flatten(output)
        return output



text_model = TextModel(model_clip.text_model, model_clip.text_projection)

In [6]:
text_input = "bike"
input_ids = processor(text=text_input, return_tensors="pt", padding="max_length", truncation = "longest_first", max_length=76)
outputs1 = text_model(input_ids["input_ids"])

text_input = "car"
input_ids = processor(text=text_input, return_tensors="pt", padding="max_length", truncation = "longest_first", max_length=76)
outputs2 = text_model(input_ids["input_ids"])

text_input = "dog jumping on a ferrari"
input_ids = processor(text=text_input, return_tensors="pt", padding="max_length", truncation = "longest_first", max_length=76)
outputs3 = text_model(input_ids["input_ids"], input_ids['attention_mask'])

In [7]:
cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
cos(outputs1, outputs2), cos(outputs1, outputs3), cos(outputs2, outputs3)


(tensor(0.8558, grad_fn=<SumBackward1>),
 tensor(0.6196, grad_fn=<SumBackward1>),
 tensor(0.6601, grad_fn=<SumBackward1>))

In [8]:
input_ids['attention_mask'].shape

torch.Size([1, 76])

In [9]:
import torch
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from torch.export import Dim, export

input_ids2 = (torch.randint(0, 20000, (1,76), dtype=torch.int64), torch.randint(0, 2, (1,76), dtype=torch.int64))


exported_program = export(text_model, input_ids2)
executorch_program = to_edge_transform_and_lower(
    exported_program,
    partitioner = [XnnpackPartitioner()]
).to_executorch()

with open("clip_text_model.pte", "wb") as file:
    file.write(executorch_program.buffer)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/przemek/anaconda3/envs/executorch-2/lib/python3.10/site-packages/torch/export/_unlift.py:81: UserWarning: Attempted to insert a get_attr Node with no underlying reference in the owning GraphModule! Call GraphModule.add_submodule to add the necessary submodule, GraphModule.add_parameter to add the necessary Parameter, or nn.Module.register_buffer to add the necessary buffer
  getattr_node = gm.graph.get_attr(lifted_node)
/Users/przemek/anaconda3/envs/executorch-2/lib/python3.10/site-packages/torch/fx/graph.py:1772: UserWarning: Node text_model_lifted_tensor_0 target text_model.lifted_tensor_0 lifted_tensor_0 of text_model does not reference an nn.Module, nn.Para

In [10]:
from executorch.runtime import Runtime

runtime = Runtime.get()

input_tensor = torch.randint(0, 20000, (1,76), dtype=torch.int64)
input_tensor2 = torch.randint(0, 2, (1,76), dtype=torch.int64)
program = runtime.load_program("clip_text_model.pte")
method = program.load_method("forward")
outputs = method.execute([input_tensor, input_tensor2])

print(outputs[0].shape, outputs[0].dtype, len(outputs), type(outputs))

[program.cpp:136] InternalConsistency verification requested but not available


torch.Size([512]) torch.float32 1 <class 'list'>


# Now vision model

In [11]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(images=image, return_tensors="pt", padding=True)

In [12]:
inputs["pixel_values"].shape, inputs["pixel_values"].dtype

(torch.Size([1, 3, 224, 224]), torch.float32)

In [13]:
class VisionModel(torch.nn.Module):
    def __init__(self, vision_model, visual_projection):
        super().__init__()
        self.vision_model = vision_model 
        self.visual_projection = visual_projection

    def forward(self, pixel_values):
        output = self.vision_model(pixel_values)
        output = output.pooler_output
        output = self.visual_projection(output)
        output = torch.flatten(output)
        return output

vision_model = TextModel(model_clip.vision_model, model_clip.visual_projection)

In [14]:
# cat
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt", padding=True)
outputs1_vis = vision_model(inputs["pixel_values"])

# bike
url = "https://farm2.staticflickr.com/1084/1148759967_49e79af6a5_z.jpg"
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt", padding=True)
outputs2_vis = vision_model(inputs["pixel_values"])

# dog
url = "https://farm8.staticflickr.com/7218/7255487026_288b791eba_z.jpg"
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt", padding=True)
outputs3_vis = vision_model(inputs["pixel_values"])

In [15]:
# outputs1 = bike, outputs3 = dog jumping on a ferrari
cos(outputs1, outputs3_vis), cos(outputs3, outputs3_vis)

(tensor(0.1535, grad_fn=<SumBackward1>),
 tensor(0.2749, grad_fn=<SumBackward1>))

In [16]:
cos(outputs1_vis, outputs3_vis), cos(outputs2_vis, outputs3_vis)

(tensor(0.6237, grad_fn=<SumBackward1>),
 tensor(0.4879, grad_fn=<SumBackward1>))

In [17]:
inputs["pixel_values"].shape, inputs["pixel_values"].dtype

(torch.Size([1, 3, 224, 224]), torch.float32)

In [18]:
input_ids3 = (torch.randn(1, 3, 224, 224, dtype=torch.float32),)

exported_program = export(vision_model, input_ids3)
executorch_program = to_edge_transform_and_lower(
    exported_program,
    partitioner = [XnnpackPartitioner()]
).to_executorch()

with open("clip_vision_model.pte", "wb") as file:
    file.write(executorch_program.buffer)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [19]:
from executorch.runtime import Runtime

runtime = Runtime.get()

input_tensor = torch.randn(1, 3, 224, 224, dtype=torch.float32)
program = runtime.load_program("clip_vision_model.pte")
method = program.load_method("forward")
outputs = method.execute([input_tensor])

print(outputs[0].shape)

[program.cpp:136] InternalConsistency verification requested but not available


torch.Size([512])


# Final test in executorch from python

In [20]:
runtime = Runtime.get()

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
input_tensor = processor(images=image, return_tensors="pt", padding=True)["pixel_values"]

program = runtime.load_program("clip_vision_model.pte")
method = program.load_method("forward")
output_vison = method.execute([input_tensor])[0]

print(output_vison.shape)

[program.cpp:136] InternalConsistency verification requested but not available


torch.Size([512])


In [21]:
text = "two cute kittens"
input_tensor = processor(text=text, return_tensors="pt", padding="max_length", truncation = "longest_first", max_length=76)
program = runtime.load_program("clip_text_model.pte")
method = program.load_method("forward")
output_text = method.execute([input_tensor["input_ids"], input_tensor["attention_mask"]])[0]

print(output_text.shape)

[program.cpp:136] InternalConsistency verification requested but not available


torch.Size([512])


In [22]:
cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
cos(output_vison, output_text)

tensor(0.2717)